In [1]:
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
adata = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop2p5/replicate/adata_500k.h5ad")

In [3]:
experiments_loop_2 = ["239", 
                      "240",
                      "248",
                      "241",
                      "242",
                      "249"]

leaveout = ["243",
            "244",
            "250",
            "245",
            "246",
            "251"]

uncertainty_assignment = ["low", "high", "high", "low", "high", "high"]

In [4]:
adata = adata[~adata.obs.experiment_number.isin(leaveout)]

In [5]:
loop_assignment = np.array(["Loop1" for _ in range(len(adata))])
loop_assignment[adata.obs["experiment_number"].isin(experiments_loop_2)] = "Loop2"

In [6]:
uncertainty_assigment = np.array(["None" for _ in range(len(adata))])
uncertainty_assigment[adata.obs["experiment_number"].isin(high_unc)] = "high_uncertainty"
uncertainty_assigment[adata.obs["experiment_number"].isin(low_unc)] = "low_uncertainty"

NameError: name 'high_unc' is not defined

In [ ]:
cols = ['gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]',
       'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]',
       'il3_[ng_ml]', 'o2_[%]', 'days_of_culture']

In [ ]:
adata.obs = adata.obs.loc[:, cols]
adata.obs = adata.obs.astype(float)

In [ ]:
adata.obs["Loops"] = loop_assignment

In [ ]:
adata_unique = adata.obs.drop_duplicates()

In [ ]:
adata_unique.loc[:, cols] = np.log1p(adata_unique.loc[:, cols])

In [ ]:
adata_unique

## Separate again loops and protocols 

In [ ]:
loops = np.array(adata_unique.loc[:, "Loops"])
protocols = adata_unique.loc[:, cols]

In [ ]:
dist_mat = np.mean((protocols.values[None, :] - protocols.values[:, None])**2, axis=-1)

np.fill_diagonal(dist_mat, np.inf)
dist_mat_pooled = dist_mat.mean(1)

dists_loop2 = dist_mat_pooled[loops=="Loop2"]

In [ ]:
dist_mat.shape

In [ ]:
sns.kdeplot(dist_mat_pooled)

unc_cols = {"high": "red", 
           "low": "blue"}

for i, d in enumerate(dists_loop2):
    plt.axvline(x=d, color=unc_cols[uncertainty_assignment[i]], linestyle='--', linewidth=1)